# CIFAR-10 — ResNet18 Fine-Tuning

Notebook version of `train.py`. Single-GPU / CPU only (no distributed training —
this matches what the script actually executes today, since the DDP wiring in
`run_distributed.sh` / `train.py` doesn't wrap the model in `DistributedDataParallel`
or shard the data, so multi-process runs were duplicating work, not parallelizing it).

Run cells top to bottom. Adjust the **Config** cell instead of passing CLI args.

## Imports

In [ ]:
import os
import csv
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from tqdm import tqdm

from sklearn.metrics import confusion_matrix

## Config

Replaces the `argparse` CLI args from `train.py`. Edit values here instead of
passing `--epochs`, `--batch-size`, etc.

In [ ]:
class Args:
    epochs = 10
    batch_size = 32
    lr = 1e-3
    data = "./data"
    output_dir = "./outputs"
    seed = 42

args = Args()

## Reproducibility

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(args.seed)

## Device

Distributed-specific logic (`RANK`/`get_rank()`) has been dropped here since this
notebook is single-process. `device` will just be your GPU if available, else CPU.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("====================================")
print("TRAINING START")
print("Device:", device)
print("====================================")

## Output Directory & Metrics CSV

In [ ]:
output_dir = Path(args.output_dir)
output_dir.mkdir(parents=True, exist_ok=True)

csv_path = output_dir / "metrics.csv"

# clean CSV each run
if csv_path.exists():
    os.remove(csv_path)

with open(csv_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["epoch", "train_loss", "train_acc", "val_loss", "val_acc"])

## Data

In [ ]:
transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406),
                         (0.229, 0.224, 0.225))
])

transform_test = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406),
                         (0.229, 0.224, 0.225))
])

train_data = datasets.CIFAR10(root=args.data, train=True, download=True, transform=transform_train)
test_data = datasets.CIFAR10(root=args.data, train=False, download=True, transform=transform_test)

train_loader = DataLoader(train_data, batch_size=args.batch_size, shuffle=True, num_workers=2)
test_loader = DataLoader(test_data, batch_size=args.batch_size, shuffle=False, num_workers=2)

## Model

ResNet18, pretrained, with `layer3`/`layer4` unfrozen and a new 10-class head.

In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

for p in model.parameters():
    p.requires_grad = False

for p in model.layer3.parameters():
    p.requires_grad = True

for p in model.layer4.parameters():
    p.requires_grad = True

model.fc = nn.Linear(model.fc.in_features, 10)
model = model.to(device)

## Loss, Optimizer, Scheduler

Discriminative learning rates: lower LR on earlier unfrozen layers, full `args.lr` on the new head.

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = optim.AdamW([
    {"params": model.layer3.parameters(), "lr": 1e-5},
    {"params": model.layer4.parameters(), "lr": 1e-4},
    {"params": model.fc.parameters(), "lr": args.lr},
], weight_decay=1e-4)

scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.epochs)

## Train / Eval Functions

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    preds_all, labels_all = [], []

    for x, y in tqdm(loader, desc="Training"):
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        preds_all.extend(torch.argmax(out, 1).cpu().numpy())
        labels_all.extend(y.cpu().numpy())

    acc = np.mean(np.array(preds_all) == np.array(labels_all))
    return total_loss / len(loader), acc

In [ ]:
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    preds_all, labels_all = [], []

    with torch.no_grad():
        for x, y in tqdm(loader, desc="Evaluating"):
            x, y = x.to(device), y.to(device)

            out = model(x)
            loss = criterion(out, y)

            total_loss += loss.item()

            preds_all.extend(torch.argmax(out, 1).cpu().numpy())
            labels_all.extend(y.cpu().numpy())

    acc = np.mean(np.array(preds_all) == np.array(labels_all))
    return total_loss / len(loader), acc, preds_all, labels_all

_Note: fixed the `evaluate()` tqdm description from `"Training"` to `"Evaluating"` — cosmetic only, no logic change._

## Training Loop

In [ ]:
best_acc = 0
preds, labels = [], []

for epoch in range(args.epochs):

    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc, preds, labels = evaluate(model, test_loader, criterion, device)

    scheduler.step()

    print(f"""
Epoch {epoch+1}/{args.epochs}
Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}
Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}
""")

    with open(csv_path, "a", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([epoch+1, train_loss, train_acc, val_loss, val_acc])

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), output_dir / "best_model.pth")
        print("Saved best model \u2714")

## Final Save & Confusion Matrix

In [ ]:
torch.save(model.state_dict(), output_dir / "last_model.pth")
print("Training complete \u2714 Best Acc:", best_acc)

cm = confusion_matrix(labels, preds)
print("Confusion Matrix:\n", cm)